In [ ]:
!pip install pandas

In [ ]:
import numpy as np

# -------------------------
# 1. Synthetic Data
# -------------------------
np.random.seed(42)
N, D_in, H, D_out = 5000, 10, 64, 3

# random inputs and target classes
X = np.random.randn(N, D_in)
true_W = np.random.randn(D_in, D_out)
y_prob = np.exp(X @ true_W) / np.sum(np.exp(X @ true_W), axis=1, keepdims=True)
y = np.array([np.random.choice(D_out, p=p) for p in y_prob])

# one-hot
Y = np.zeros((N, D_out))
Y[np.arange(N), y] = 1

# -------------------------
# 2. Initialize parameters
# -------------------------
W1 = np.random.randn(D_in, H) / np.sqrt(D_in)
b1 = np.zeros((1, H))
W2 = np.random.randn(H, D_out) / np.sqrt(H)
b2 = np.zeros((1, D_out))

# -------------------------
# 3. Helper functions
# -------------------------
def relu(z): return np.maximum(0, z)
def relu_deriv(z): return (z > 0).astype(float)
def softmax(z): return np.exp(z - np.max(z, axis=1, keepdims=True)) / np.exp(z - np.max(z, axis=1, keepdims=True)).sum(axis=1, keepdims=True)

def cross_entropy(pred, target):
    return -np.mean(np.sum(target * np.log(pred + 1e-9), axis=1))

# -------------------------
# 4. Training Loop
# -------------------------
lr = 0.01
epochs = 200
batch_size = 64

for epoch in range(epochs):
    perm = np.random.permutation(N)
    X, Y = X[perm], Y[perm]
    total_loss = 0

    for i in range(0, N, batch_size):
        xb = X[i:i+batch_size]
        yb = Y[i:i+batch_size]

        # forward
        z1 = xb @ W1 + b1
        a1 = relu(z1)
        z2 = a1 @ W2 + b2
        y_pred = softmax(z2)

        # loss
        loss = cross_entropy(y_pred, yb)
        total_loss += loss * len(xb)

        # backward
        dz2 = (y_pred - yb) / len(xb)
        dW2 = a1.T @ dz2
        db2 = np.sum(dz2, axis=0, keepdims=True)
        da1 = dz2 @ W2.T
        dz1 = da1 * relu_deriv(z1)
        dW1 = xb.T @ dz1
        db1 = np.sum(dz1, axis=0, keepdims=True)

        # update
        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2

    if epoch % 10 == 0:
        logits = relu(X @ W1 + b1) @ W2 + b2
        preds = np.argmax(logits, axis=1)
        acc = np.mean(preds == np.argmax(Y, axis=1))
        print(f"Epoch {epoch:03d} | Loss: {total_loss/N:.4f} | Acc: {acc:.4f}")

# -------------------------
# 5. Inference on Random Sample
# -------------------------
sample = np.random.randn(1, D_in)
out = softmax(relu(sample @ W1 + b1) @ W2 + b2)
print("\nRandom sample prediction probabilities:", out)
print("Predicted class:", np.argmax(out))
